In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wordcloud import WordCloud, STOPWORDS

In [ ]:
%pip install scikit-learn pandas
%pip install pyarrow

In [ ]:
%pwd
df = pd.read_parquet("../DataFiles/sampleddata.parquet")
print(df.shape)
df.head()

In [ ]:
df.describe()

In [ ]:
df.groupby('week')[['retweetCount','likeCount']].describe()

In [ ]:
df['cleanedTweet']

In [ ]:
weekly_words = df.groupby('week')['cleanedTweet'].agg(' '.join)
stopwords = set(STOPWORDS)

for index, week in enumerate(weekly_words):
    # Generate the word cloud
    wordcloud = WordCloud(
        width=600,
        height=300,
        background_color='white',
        stopwords=set(STOPWORDS)
    ).generate(week)

    # Display the word cloud labeled by the week ending date
    plt.figure(figsize=(8, 4))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.title(f"Word Cloud for Week: {index}", fontsize=14)
    plt.axis('off')
    plt.show()

In [ ]:
df['tweet_length'] = df['cleanedTweet'].str.split().str.len()
grouped = df.groupby('week')

df.hist(column='tweet_length', by='week', edgecolor='black',figsize=(10,10))
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('week')['tweet_length'].describe()

## Topic Modeling: LDA & NMF (TKG)


FYI NOTES: This portion of the notebook used unsupervised topic modeling (LDA and NMF) on the cleaned
tweets to identify the main discussion themes and track how they shift week
to week during the campaign.

In [ ]:
#Uses Jesse's cleanedTweet column to drop missing values. 
docs=df["cleanedTweet"].dropna().astype(str)
#Creating a new dataframe with only tweets that have 5 or more words for meaningful for analysis.
docs=docs[docs.str.split().str.len() >= 5]
print(len(docs), "tweets used for topic modeling")
docs.head(10)

In [ ]:
#Fit LDA and NMF models to the cleaned tweets. This will help identify underlying topics in the tweets.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF

# Number of topics for topic modeling. This can be adjusted based on the desired granularity of topics.
N_TOPICS = 5

# LDA (Latent Dirichlet Allocation)
# max_df=0.95 ignores words that appear in more than 95% of tweets;
# min_df=5 ignores words that appear in fewer than 5 tweets
count_vec = CountVectorizer(stop_words="english", max_df=0.95, min_df=5)
X_counts = count_vec.fit_transform(docs)
lda = LatentDirichletAllocation(n_components=N_TOPICS, random_state=42)
lda.fit(X_counts)

#NMF (Non-negative Matrix Factorization)
tfidf_vec = TfidfVectorizer(stop_words="english", max_df=0.95, min_df=5)
X_tfidf = tfidf_vec.fit_transform(docs)
nmf = NMF(n_components=N_TOPICS, init="nndsvd", random_state=42, max_iter=500)
nmf.fit(X_tfidf)





In [ ]:
#TOPICS
# Prints the top words for each topic so we can interpret what each one is about
def show_topics(model, vectorizer, n_words=10):
    words = vectorizer.get_feature_names_out()
    for i, topic in enumerate(model.components_):
        #Sorts words by how strongly they belong to this topic and keep the top n
        top = [words[j] for j in topic.argsort()[-n_words:][::-1]]
        print(f"Topic {i}: {', '.join(top)}")

# Compare the two models: which gives clearer, less overlapping topics?
print("--- LDA---")
show_topics(lda, count_vec)
print("\n--- NMF ---")
show_topics(nmf, tfidf_vec)

In [ ]:
#Topics Over Time
# How much each of the NMF topics shows up in each week of the campaign;
# rising/falling lines show which narratives gained or lost attention
doc_topics = nmf.transform(X_tfidf)              
topic_df = pd.DataFrame(doc_topics, index=docs.index)
topic_df["week"] = df.loc[docs.index, "week"]    

weekly_topics = topic_df.groupby("week").mean()
weekly_topics.columns = [
    "Biden",
    "Trump",
    "Harris",
    "MAGA movement",
    "Partisan talk",
]
weekly_topics.plot(figsize=(10, 5), title="NMF topic share by week")

At 5 topics, both models grouped tweets by candidate (Biden, Trump, MAGA, Harris, partisan talk) instead of by issue. To check whether more topics would surface issue topics, I reran LDA and NMF at 20 and 30 topics. Candidate names, party labels, and filler words were added to the stop words, and "max_df" was lowered to 0.5 so very common words are dropped.

(These runs are exploritory & separate from the 5 topic NMF above, which the rest of the notebook still uses.)

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Candidate names, party labels, and filler words show up in so many tweets that they
# take over the topics, so we remove them to let issue words come through
name_stops = {
    "trump", "trumps", "donald", "biden", "bidens", "joe", "harris", "harriss", "kamala", "kamalas",
    "maga", "vance", "walz", "jd", "tim", "president", "election", "vote", "voting",
    "democrats", "republicans", "democrat", "republican", "gop",
    "amp", "don", "didn", "just", "like", "did", "know", "let", "said", "people", "doesn", "isn",
    "going", "think", "want", "right", "need", "make", "way", "say", "really", "ve", "ll",
    "good", "time", "got", "does",
}
k_stops = list(ENGLISH_STOP_WORDS.union(name_stops))

# max_df=0.5 drops words that appear in more than half the tweets
count_vec_k = CountVectorizer(stop_words=k_stops, max_df=0.5, min_df=5)
X_counts_k = count_vec_k.fit_transform(docs)
tfidf_vec_k = TfidfVectorizer(stop_words=k_stops, max_df=0.5, min_df=5)
X_tfidf_k = tfidf_vec_k.fit_transform(docs)

# Fit LDA and NMF at 20 and 30 topics (same random_state so results are repeatable)
topic_runs = {}
for k in [20, 30]:
    topic_runs[("LDA", k)] = LatentDirichletAllocation(n_components=k, random_state=42).fit(X_counts_k)
    topic_runs[("NMF", k)] = NMF(n_components=k, init="nndsvd", random_state=42, max_iter=500).fit(X_tfidf_k)

for (name, k), model in topic_runs.items():
    print(f"\n--- {name}, {k} topics ---")
    show_topics(model, count_vec_k if name == "LDA" else tfidf_vec_k)

Findings:

- 20 topics: only a few topics were issue related, and they mixed several issues together (for example, NMF gave one topic with border, israel, war, immigration, and inflation).
- 30 topics: LDA separated out clearer issue topics: economy (inflation, economy, jobs, prices, gas, cost), immigration (border, illegal, illegals), foreign policy (war, russia, ukraine; and israel, hamas, netanyahu), a mixed abortion topic (women, rights, abortion), and courts (supreme court). NMF at 30 changed little from 20.
- Most other topics are  campaignevents (RFK Jr., the DNC and RNC, the assassination attempt), reactions, and hashtags rather than issues.

The main reason is the data itself, only a small amount of the tweets mention any issue (see the issue coverage check below). Adding topics mostly splits the non issue chatter into smaller pieces instead of surfacing new issues. This is why the next section uses keywords to pull the issue tweets first, then runs NMF within each issue to track how its framing shifts.

### Topic Matching and Shift Scoring
This portion of the notebook we topic matched by pairing each period's topics with the next period's most similar topics. secondly, we do shift scoring to measure how much the issue's framing changed between periods.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import NMF
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import jensenshannon

print(df.shape)
print(df.columns.tolist())

In [ ]:
# Keywords used to consolidate tweet on a given issue.
ISSUE_KEYWORDS = {
    "immigration": r"immigra|border|migrant|asylum|deport|illegals|illegal alien|refugee",
    "economy":     r"econom|inflation|prices|jobs|tax|wages|gas price|groceries",
    "abortion":    r"abortion|\broe\b|roevember|restoreroe|pro.?life|pro.?choice|reproductive|ivf",
    "democracy":   r"democracy|january 6|jan 6|insurrection|stolen election|fascist",
    "foreign_policy": r"ukraine|russia|israel|gaza|china|\bnato\b"}
text = df["cleanedTweet"].fillna("").str.lower()

# One True/False column per issue, named issue_<name> (a tweet can mention more than one issue)
for issue, pattern in ISSUE_KEYWORDS.items():
    df[f"issue_{issue}"] = text.str.contains(pattern, regex=True)

# How many tweets mention each issue
print(df[[f"issue_{i}" for i in ISSUE_KEYWORDS]].sum().sort_values(ascending=False))

In [ ]:
# Issue coverage: what share of all tweets mention at least one issue
any_issue = df[[f"issue_{i}" for i in ISSUE_KEYWORDS]].any(axis=1)
print(f"{any_issue.sum()} of {len(df)} tweets ({any_issue.mean():.1%}) mention at least one issue")

In [ ]:
ISSUE = "immigration"   # test issue; can be changed to any issue in ISSUE_KEYWORDS
PERIOD_WEEKS = 1        
K = 5                   # number of topics (frames) per period

# tweets about specific issue
issue_df = df[df[f"issue_{ISSUE}"]].copy()
issue_df = issue_df[issue_df["cleanedTweet"].str.split().str.len() >= 5].reset_index(drop=True)

#Assigning issues a period (each week has its own period)
issue_df["period"] = issue_df["week"] // PERIOD_WEEKS

print(len(issue_df), f"tweets about {ISSUE}")
print(issue_df["period"].value_counts().sort_index())  

In [ ]:
# Stop typical stopwords/filler words
custom_stops = list(ENGLISH_STOP_WORDS.union(
    {"amp", "don", "didn", "just", "like", "did", "know", "let", "said", "people"}
))

# One shared vocabulary so topics from different periods can be compared
vec = TfidfVectorizer(stop_words=custom_stops, max_df=0.95, min_df=2)
X = vec.fit_transform(issue_df["cleanedTweet"])
words = vec.get_feature_names_out()

def top_words(topic_row, n=8):
    """Top n words for one topic."""
    return ", ".join(words[i] for i in topic_row.argsort()[-n:][::-1])

models = {}
for period, rows in sorted(issue_df.groupby("period").indices.items()):
    X_p = X[rows]
    if X_p.shape[0] < K * 10:   # skip periods with too few tweets
        print(f"Period {period}: only {X_p.shape[0]} tweets, skipped")
        continue
    period_nmf = NMF(n_components=K, init="nndsvd", random_state=42, max_iter=500)
    W = period_nmf.fit_transform(X_p)          # topic scores for each tweet
    props = W.sum(axis=0) / W.sum()            # each topic's share of the period
    models[period] = {"H": period_nmf.components_, "props": props}

    print(f"\n=== Period {period} ({X_p.shape[0]} tweets) ===")

    for t, row in enumerate(period_nmf.components_):
        print(f"  Topic {t} ({props[t]:.0%}): {top_words(row)}")
        top_words_per_period.append([period,t,top_words(row,20)])
    pd.DataFrame(top_words_per_period).to_csv("../DataFiles/top_words_per_period.csv",index=False,header=['Period','Topic','Words'])

In [ ]:
#Jesse adapting TK's code to find top words per each issue per period
ISSUES = ["immigration","foreign_policy","economy","democracy","abortion"]   # list of all issues
PERIOD_WEEKS = 1
K = 1                   # Setting to single topic for each issue for clearer analysis
all_issues = pd.DataFrame()
# tweets about specific issue
for issue in ISSUES:
    issue_df = df[df[f"issue_{issue}"]].copy()
    issue_df = issue_df[issue_df["cleanedTweet"].str.split().str.len() >= 5].reset_index(drop=True)
    #Assigning issues a period (each week has its own period)
    issue_df["period"] = issue_df["week"] // PERIOD_WEEKS
    vec = TfidfVectorizer(stop_words=custom_stops, max_df=0.95, min_df=2)
    X = vec.fit_transform(issue_df["cleanedTweet"])
    words = vec.get_feature_names_out()
    top_words_per_period = []
    for period, rows in sorted(issue_df.groupby("period").indices.items()):
        X_p = X[rows]
        if X_p.shape[0] < K * 10:   # skip periods with too few tweets
            print(f"Period {period}: only {X_p.shape[0]} tweets, skipped")
            continue
        period_nmf = NMF(n_components=K, init="nndsvd", random_state=42, max_iter=500)
        W = period_nmf.fit_transform(X_p)          # topic scores for each tweet
        props = W.sum(axis=0) / W.sum()            # each topic's share of the period
        models[period] = {"H": period_nmf.components_, "props": props}
        for t, row in enumerate(period_nmf.components_):
            top_words_per_period.append([issue,period,t,top_words(row,20)])
        temp_df = pd.DataFrame(top_words_per_period,columns=["Issue","period","Topic","Words"])
    all_issues= pd.concat([all_issues,temp_df],ignore_index=True)
all_issues.to_csv("../DataFiles/top_words_per_period.csv",index=False,
                                          header=['Issue','Period','Topic','Words'])

In [ ]:
#matching topics across periods
def match_topics(H_a, H_b, threshold=0.2):
    """
    Pair each topic in period A with its most similar topic in period B using cosine similarity of their word weights (1 = identical, 0 = no overlap).
    Pairs below the threshold are "unmatched": a frame that appeared or disappeared.
    """
    sim = cosine_similarity(H_a, H_b)
    rows, cols = linear_sum_assignment(-sim)   # best one-to-one pairing
    return [(a, b, sim[a, b], sim[a, b] >= threshold) for a, b in zip(rows, cols)]

In [ ]:
periods = sorted(models)
results = []
for p_a, p_b in zip(periods[:-1], periods[1:]):
    A, B = models[p_a], models[p_b]
    pairs = match_topics(A["H"], B["H"])

    # Content shift: how much matched topics changed wording (0 = same, 1 = totally different)
    matched = [s for _, _, s, ok in pairs if ok]
    content_shift = np.mean([1 - s for s in matched]) if matched else 1.0

    # Proportion shift: how much attention moved between topics (0 = same mix, 1 = completely different)
    a_idx = [a for a, _, _, _ in pairs]
    b_idx = [b for _, b, _, _ in pairs]
    proportion_shift = jensenshannon(A["props"][a_idx], B["props"][b_idx], base=2)

    results.append({
        "from_period": p_a, "to_period": p_b,
        "content_shift": round(content_shift, 3),
        "proportion_shift": round(proportion_shift, 3),
        "new_frames": sum(1 for _, _, _, ok in pairs if not ok),
        "shift_score": round((content_shift + proportion_shift) / 2, 3),
    })

    # Show how each topic's words changed
    print(f"\n--- Period {p_a} -> Period {p_b} ---")
    for a, b, s, ok in pairs:
        label = "matched" if ok else "NEW / DROPPED"
        print(f"  [{label}, similarity {s:.2f}]")
        print(f"     before: {top_words(A['H'][a])}")
        print(f"     after:  {top_words(B['H'][b])}")

shift_df = pd.DataFrame(results)
shift_df

In [ ]:
# Peaks=periods where the issue was reframed the most
ax = shift_df.plot(x="to_period", y=["content_shift", "proportion_shift", "shift_score"], marker="o", figsize=(9, 4), title=f"Narrative shift scores: {ISSUE}")
ax.set_xlabel(f"Period ({PERIOD_WEEKS}-week blocks)")
ax.set_ylabel("Shift (0 = no change, 1 = complete change)")

In [ ]:
# Feature Table for Supervised Models (one row per week per issue)
def shift_between(A, B, threshold=0.2):
    pairs = match_topics(A["H"], B["H"], threshold)

    # Content shift: how much matched topics changed wording (0=same, 1=totally different)
    matched = [s for _, _, s, ok in pairs if ok]
    content_shift = np.mean([1 - s for s in matched]) if matched else 1.0

    # Proportion shift: how much attention moved between topics (0=same mix, 1=completely different)
    a_idx = [a for a, _, _, _ in pairs]
    b_idx = [b for _, b, _, _ in pairs]
    proportion_shift = jensenshannon(A["props"][a_idx], B["props"][b_idx], base=2)

    return {
        "content_shift": round(content_shift, 3),
        "proportion_shift": round(proportion_shift, 3),
        "new_frames": sum(1 for _, _, _, ok in pairs if not ok),
        "shift_score": round((content_shift + proportion_shift) / 2, 3),}


def run_issue(issue, period_weeks=PERIOD_WEEKS, k=K, threshold=0.2):
    """
    #Runs the pilot pipeline above (per period NMF + topic matching) for any issue, without printing.
    Returns one row per period with the shift scores for moving INTO that period from the
    previous fitted period. The first period has nothing to compare to, so it gets no row.
    """
    # Tweets about this issue with at great than or equal to 5 words, grouped into periods
    issue_tweets = df[df[f"issue_{issue}"]].copy()
    issue_tweets = issue_tweets[issue_tweets["cleanedTweet"].str.split().str.len() >= 5].reset_index(drop=True)
    issue_tweets["period"] = issue_tweets["week"] // period_weeks

    # One shared vocabulary per issue so the periods can be compared
    issue_vec = TfidfVectorizer(stop_words=custom_stops, max_df=0.95, min_df=2)
    X_issue = issue_vec.fit_transform(issue_tweets["cleanedTweet"])

    issue_models = {}
    for period, rows in sorted(issue_tweets.groupby("period").indices.items()):
        if len(rows) < k * 10:   # skip periods with too few tweets (same rule as the pilot)
            continue
        period_nmf = NMF(n_components=k, init="nndsvd", random_state=42, max_iter=500)
        W = period_nmf.fit_transform(X_issue[rows])
        issue_models[period] = {"H": period_nmf.components_, "props": W.sum(axis=0) / W.sum()}

    fitted = sorted(issue_models)
    rows = [{"period": p_b, "from_period": p_a, **shift_between(issue_models[p_a], issue_models[p_b], threshold)}
            for p_a, p_b in zip(fitted[:-1], fitted[1:])]
    columns = ["period", "from_period", "content_shift", "proportion_shift", "new_frames", "shift_score"]
    return pd.DataFrame(rows, columns=columns)
run_issue("immigration")

In [ ]:
ISSUES = list(ISSUE_KEYWORDS)

# 1) Tweet counts: one row per week x issue
issue_cols = [f"issue_{i}" for i in ISSUES]
features = (
    df.groupby("week")[issue_cols].sum()
      .rename(columns=lambda c: c.removeprefix("issue_"))
      .reset_index()
      .melt(id_vars="week", var_name="issue", value_name="n_tweets_issue")
)

# 2) Week info: calendar dates (weeks are 7-day blocks starting Wed May 1, 2024),
#    total tweets, and how many days actually have data (weeks 13, 16, 17 have gaps)
first_day = pd.to_datetime(df["date"]).min()
week_info = df.groupby("week").agg(n_tweets_week=("week", "size"), days_with_data=("date", "nunique")).reset_index()
week_info["week_start"] = (first_day + pd.to_timedelta(week_info["week"] * 7, unit="D")).dt.date
week_info["week_end"] = (first_day + pd.to_timedelta(week_info["week"] * 7 + 6, unit="D")).dt.date
features = features.merge(week_info, on="week")

# 3) Issue share: what fraction of that week's tweets mention the issue, and its weekly change
features = features.sort_values(["issue", "week"]).reset_index(drop=True)
features["issue_share"] = features["n_tweets_issue"] / features["n_tweets_week"]
features["issue_share_change"] = features.groupby("issue")["issue_share"].diff()   # blank in week 0

# 4) Shift scores: every week in a period gets that periods scores (blank in each issue's first period
#    and in periods with too few tweets to fit topics)
features["period"] = features["week"] // PERIOD_WEEKS
all_shifts = pd.concat([run_issue(i).assign(issue=i) for i in ISSUES], ignore_index=True)
features = features.merge(all_shifts, on=["issue", "period"], how="left")

# 5) Topic shares: among this issue's tweets that week, the average share of each overall NMF topic
#    (Biden, Trump, ...). These topics have the same meaning every week, unlike the per-period topics.
topic_cols = ["topic_" + name.lower().replace(" ", "_") for name in weekly_topics.columns]
shares = pd.DataFrame(doc_topics, index=docs.index, columns=topic_cols)
shares = shares.div(shares.sum(axis=1), axis=0)   # each tweet's topic weights add up to 1
shares["week"] = df.loc[docs.index, "week"]
weekly_shares = pd.concat(
    [shares[df.loc[docs.index, f"issue_{i}"]].groupby("week")[topic_cols].mean().reset_index().assign(issue=i)
     for i in ISSUES],
    ignore_index=True,
)
features = features.merge(weekly_shares, on=["week", "issue"], how="left")

features = features[
    ["week", "week_start", "week_end", "issue", "n_tweets_issue", "n_tweets_week", "days_with_data",
     "issue_share", "issue_share_change", "period", "from_period", "content_shift", "proportion_shift",
     "new_frames", "shift_score"] + topic_cols
]
features.to_csv("../DataFiles/tk_issue_week_features.csv", index=False)
print(features.shape, "-> saved to DataFiles/tk_issue_week_features.csv")
features.head(12)

In [ ]:
# Shared settings for all Week 4 charts: one fixed color per issue, used in every chart
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

ISSUE_ORDER = ["foreign_policy", "economy", "immigration", "democracy", "abortion"]  # most to least tweets
ISSUE_LABELS = {"foreign_policy": "Foreign policy", "economy": "Economy", "immigration": "Immigration",
                "democracy": "Democracy", "abortion": "Abortion"}
ISSUE_COLORS = {"foreign_policy": "#2a78d6", "economy": "#eb6834", "immigration": "#1baf7a",
                "democracy": "#eda100", "abortion": "#e87ba4"}

plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.titleweight": "bold", "axes.titlesize": 13, "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e", "legend.frameon": False,
})

features["week_start"] = pd.to_datetime(features["week_start"])
score_cols = ["issue_share", "issue_share_change", "content_shift", "proportion_shift", "new_frames", "shift_score"]
features[score_cols] = features[score_cols].apply(pd.to_numeric, errors="coerce")  # blanks become NaN

In [ ]:
# VISUAL 1: How much attention each issue got, week by week
share = features.pivot(index="week_start", columns="issue", values="issue_share")[ISSUE_ORDER] * 100

fig, ax = plt.subplots(figsize=(11, 5))
for issue in ISSUE_ORDER:
    ax.plot(share.index, share[issue], color=ISSUE_COLORS[issue], linewidth=2, label=ISSUE_LABELS[issue])
ax.set_title("Share of weekly tweets mentioning each issue", pad=28)
ax.set_ylabel("% of that week's tweets")
ax.set_xlabel("Week starting")
ax.set_ylim(bottom=0)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=5)
fig.tight_layout()
plt.show()

In [ ]:
# VISUAL 2: Heatmap of weekly shift scores (how much each issue's framing changed from the week before)
from matplotlib.colors import LinearSegmentedColormap

blues = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#86b6ef", "#2a78d6", "#184f95", "#0d366b"])
blues.set_bad("#f0efec")  # weeks with no score (first week, or too few tweets)

heat = features.pivot(index="issue", columns="week", values="shift_score").loc[ISSUE_ORDER]
week_dates = features.drop_duplicates("week").set_index("week")["week_start"]

fig, ax = plt.subplots(figsize=(13, 3.6))
vals = heat.values.astype(float)
im = ax.imshow(vals, aspect="auto", cmap=blues, vmin=np.nanmin(vals), vmax=np.nanmax(vals))  # color range = actual score range
ax.set_yticks(range(len(ISSUE_ORDER)), [ISSUE_LABELS[i] for i in ISSUE_ORDER])
tick_weeks = heat.columns[::4]
ax.set_xticks([list(heat.columns).index(w) for w in tick_weeks],
              [week_dates[w].strftime("%b %d") for w in tick_weeks])
ax.grid(False)
ax.set_title("Weekly narrative shift score by issue (darker = bigger reframing)")
cbar = fig.colorbar(im, ax=ax, pad=0.01)
cbar.set_label("Shift score (higher = more reframing)")
fig.tight_layout()
plt.show()

In [ ]:
# VISUAL 3: Shift score over time, one small chart per issue, with each issue's top 3 weeks marked
fig, axes = plt.subplots(len(ISSUE_ORDER), 1, figsize=(11, 10), sharex=True, sharey=True)
for ax, issue in zip(axes, ISSUE_ORDER):
    s = features[features["issue"] == issue].set_index("week_start")["shift_score"]
    ax.plot(s.index, s.values, color=ISSUE_COLORS[issue], linewidth=2, marker="o", markersize=4)
    ax.axhline(s.mean(), color="#898781", linewidth=1, linestyle="--")
    top3 = s.nlargest(3)
    ax.scatter(top3.index, top3.values, s=60, color=ISSUE_COLORS[issue], edgecolor="#0b0b0b", zorder=3)
    for d, v in top3.items():
        ax.annotate(d.strftime("%b %d"), (d, v), xytext=(0, 7), textcoords="offset points",
                    ha="center", fontsize=8, color="#52514e")
    ax.set_ylabel(ISSUE_LABELS[issue], rotation=0, ha="right", va="center")
axes[0].set_title("Weekly shift score by issue (dashed line = issue average; dots = top 3 weeks)")
axes[-1].set_xlabel("Week starting") 
fig.tight_layout()
plt.show()

Topic-issue mapping (LDA, 20 topics, w/ full 465k sample)

From the 20 topic LDA run above (candidate names removed). Each topic was read by its top words and assigned to one of our 5 issues, the other 14 topics are candidates, media, reactions, and hashtags rather than issues.

The topic model found the same issues our keywords target, which supports using keywords to pull issue tweets. Healthcare never appeared as a topic at 20 or 30, so it was dropped.

In [ ]:
#  LDA vs NMF: Topic Quality
#Comparing LDA and NMF with topic coherence (NPMI: higher means a topic's top words tend to appear in the same tweets), topic diversity (share of unique words across topics), and how many of our 5 issues show up in a topic's top words.
import re
from matplotlib.patches import Patch


def npmi_coherence(model, binary_matrix, n_words=10):
    n_docs = binary_matrix.shape[0]
    scores = []
    for topic in model.components_:
        idx = topic.argsort()[-n_words:][::-1]
        sub = binary_matrix[:, idx]
        co = (sub.T @ sub).toarray() / n_docs
        p = np.diag(co)
        pair_scores = []
        for a in range(n_words):
            for b in range(a + 1, n_words):
                if co[a, b] == 0:
                    pair_scores.append(-1.0)
                else:
                    pmi = np.log(co[a, b] / (p[a] * p[b]))
                    pair_scores.append(pmi / -np.log(co[a, b]))
        scores.append(np.mean(pair_scores))
    return np.array(scores)


def top_words(model, vectorizer, n_words=10):
    words = vectorizer.get_feature_names_out()
    return [[words[j] for j in t.argsort()[-n_words:][::-1]]
            for t in model.components_]


bin_counts = (X_counts > 0).astype(np.float32)
bin_counts_k = (X_counts_k > 0).astype(np.float32)

runs = [("LDA", 5, lda, count_vec, bin_counts),
        ("NMF", 5, nmf, tfidf_vec, bin_counts)]
for (name, k), model in topic_runs.items():
    vec = count_vec_k if name == "LDA" else tfidf_vec_k
    runs.append((name, k, model, vec, bin_counts_k))

rows = []
for name, k, model, vec, binary in runs:
    words = top_words(model, vec)
    joined = [" ".join(w) for w in words]
    hit = {i: [bool(re.search(pat, t)) for t in joined]
           for i, pat in ISSUE_KEYWORDS.items()}
    rows.append({
        "Model": name,
        "Topics": k,
        "Coherence (NPMI)": round(npmi_coherence(model, binary).mean(), 3),
        "Topic diversity": round(len({w for t in words for w in t})
                                 / (10 * k), 2),
        "Topics with an issue word": sum(any(h[j] for h in hit.values())
                                         for j in range(k)),
        "Issues covered (of 5)": sum(any(v) for v in hit.values()),
    })

comparison = pd.DataFrame(rows).set_index(["Model", "Topics"])
display(comparison)

In [ ]:
# LDA topics we read as each issue from their top words (random_state=42, so topic numbers are repeatable)
lda20 = topic_runs[("LDA", 20)]
vocab = count_vec_k.get_feature_names_out()
weights = lda20.components_ / lda20.components_.sum(axis=1, keepdims=True)
best = {"foreign_policy": 17, "economy": 15, "immigration": 7,
        "democracy": 4, "abortion": 13}

fig, axes = plt.subplots(1, len(best), figsize=(3.2 * len(best), 3.6),
                         sharex=True)
for ax, (issue, t) in zip(axes, best.items()):
    top = weights[t].argsort()[-8:]
    ax.barh(vocab[top], weights[t][top], color=ISSUE_COLORS[issue])
    ax.set_title(f"{ISSUE_LABELS[issue]} (topic {t})", fontsize=10)
    ax.grid(False)
fig.supxlabel("Share of the topic's word weight")
fig.tight_layout()
plt.show()


In [ ]:
# How many tweets each LDA topic dominates, colored by issue (gray = not an issue)
doc_topics = lda20.transform(X_counts_k)
share = np.bincount(doc_topics.argmax(axis=1), minlength=20) / len(doc_topics) * 100
labels = [f"{t}: " + ", ".join(w[:3])
          for t, w in enumerate(top_words(lda20, count_vec_k))]
topic_issue = {t: i for i, t in best.items()}
order = share.argsort()
colors = [ISSUE_COLORS[topic_issue[t]] if t in topic_issue else "#c3c2b7"
          for t in order]

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh([labels[t] for t in order], share[order], color=colors)
ax.set_xlabel("% of tweets where this is the strongest topic")
ax.set_title("LDA, 20 topics: share of tweets by topic", pad=28)
ax.legend(handles=[Patch(color=ISSUE_COLORS[i], label=ISSUE_LABELS[i])
                   for i in best] + [Patch(color="#c3c2b7", label="Not an issue")],
          loc="lower left", bbox_to_anchor=(0, 1.0), ncol=6, fontsize=8)
ax.grid(False)
fig.tight_layout()
plt.show()

Top 20 words for every topic in the LDA and NMF runs at 5, 20 and 30 topics, saved to `DataFiles/tk_topic_top_words.csv`

In [ ]:
#Top 20 words for every topic in the LDA and NMF runs at 5, 20 and 30 topics,
#saved to `DataFiles/tk_topic_top_words.csv`
rows = []
all_runs = [("LDA", 5, lda, count_vec), ("NMF", 5, nmf, tfidf_vec)]
for (name, k), model in topic_runs.items():
    all_runs.append((name, k, model, count_vec_k if name == "LDA" else tfidf_vec_k))

for name, k, model, vec in all_runs:
    for t, words in enumerate(top_words(model, vec, n_words=20)):
        rows.append({"model": name, "n_topics": k, "topic": t,
                     "top_words": " ".join(words)})

OUT = "../DataFiles/tk_topic_top_words.csv"
pd.DataFrame(rows).to_csv(OUT, index=False)
print(f"Saved {len(rows)} topics to {OUT}")

### Sensitivity Analysis
Do the shift scores hold up if we change the settings? I rerun the weekly shift scoring with different choices and compare to our baseline (K = 5 frames per week, match threshold = 0.2, 1-week periods):
- K = 3 or 7:  fewer or more frames per week
- Threshold = 0.1 or 0.3: looser or stricter rule for when two frames count as "the same"
- 2 week periods: coarser time block
- If same issues stay on top and the weekly patterns line up with the baseline, our findings don't depend on these choices.

In [ ]:
# SENSITIVITY: rerun shift scoring under different settings (takes a few minutes)
def scores_for(k=5, threshold=0.2, period_weeks=1):
    return pd.concat([run_issue(i, period_weeks=period_weeks, k=k, threshold=threshold).assign(issue=i)
                      for i in ISSUE_ORDER], ignore_index=True)

settings = {
    "Baseline (K=5, t=0.2, 1 wk)": dict(),
    "K = 3": dict(k=3),
    "K = 7": dict(k=7),
    "Threshold = 0.1": dict(threshold=0.1),
    "Threshold = 0.3": dict(threshold=0.3),
    "2-week periods": dict(period_weeks=2),
}
runs = {name: scores_for(**kw) for name, kw in settings.items()}
base = runs["Baseline (K=5, t=0.2, 1 wk)"]
base_mean = base.groupby("issue")["shift_score"].mean()

rows = []
for name, r in runs.items():
    mean_by_issue = r.groupby("issue")["shift_score"].mean().reindex(ISSUE_ORDER)
    row = {"Setting": name}
    row.update({ISSUE_LABELS[i]: round(mean_by_issue[i], 3) for i in ISSUE_ORDER})
    # Do the issues rank in the same order (most to least reframing) as the baseline?
    row["Issue ranking vs baseline (Spearman)"] = round(mean_by_issue.corr(base_mean.reindex(ISSUE_ORDER), method="spearman"), 2)
    # Week-to-week pattern: same periods only (not comparable for 2-week periods)
    if settings[name].get("period_weeks", 1) == 1:
        merged = r.merge(base, on=["issue", "period"], suffixes=("", "_base"))
        corrs = merged.groupby("issue").apply(lambda g: g["shift_score"].corr(g["shift_score_base"], method="spearman"),
                                              include_groups=False)
        row["Weekly pattern vs baseline (median Spearman)"] = round(corrs.median(), 2)
    else:
        row["Weekly pattern vs baseline (median Spearman)"] = "n/a (different periods)"
    rows.append(row)

sensitivity = pd.DataFrame(rows).set_index("Setting")
sensitivity

Topic count (5, 20, 30): see "Testing More Topics" above. At 5 topics the models grouped tweets by candidate; at 20 (LDA) issue topics appeared; 30 mostly split those issues into smaller pieces.

### Results Analysis

In [ ]:
# RESULTS: summary of each issue's weekly shift scores, and its 3 biggest reframing weeks
summary_rows = []
for issue in ISSUE_ORDER:
    f = features[features["issue"] == issue].dropna(subset=["shift_score"])
    top3 = f.nlargest(3, "shift_score")
    summary_rows.append({
        "Issue": ISSUE_LABELS[issue],
        "Total tweets": int(features.loc[features["issue"] == issue, "n_tweets_issue"].sum()),
        "Weeks scored": len(f),
        "Avg shift score": round(f["shift_score"].mean(), 3),
        "Avg content shift": round(f["content_shift"].mean(), 3),
        "Avg proportion shift": round(f["proportion_shift"].mean(), 3),
        "Top 3 weeks (shift score)": ", ".join(f"{d:%b %d} ({v:.2f})" for d, v in zip(top3["week_start"], top3["shift_score"])),
        # Does reframing happen in the same weeks attention changes? (+1 = always together, 0 = unrelated)
        "Shift vs attention change (Spearman)": round(f["shift_score"].corr(f["issue_share_change"].abs(), method="spearman"), 2),
    })
results = pd.DataFrame(summary_rows).set_index("Issue")
results

### Stream Plot: Issue Attention Over the Campaign

Each band is one issue; its thickness is the number of tweets mentioning that issue that week. Bands are centered so the overall shape shows how total issue talk rose and fell, and each band's width shows its share.

In [ ]:
# STREAM PLOT: weekly tweet counts per issue, stacked around a centered baseline
counts = features.pivot(index="week_start", columns="issue", values="n_tweets_issue")[ISSUE_ORDER]

fig, ax = plt.subplots(figsize=(12, 5.5))
ax.stackplot(counts.index, [counts[i] for i in ISSUE_ORDER], baseline="wiggle",
             colors=[ISSUE_COLORS[i] for i in ISSUE_ORDER], labels=[ISSUE_LABELS[i] for i in ISSUE_ORDER],
             edgecolor="#fcfcfb", linewidth=1)

# Campaign events as reference lines
events = {"Debate": "2024-06-27", "Assassination attempt": "2024-07-13", "Biden exits": "2024-07-21",
          "DNC": "2024-08-19", "Harris–Trump debate": "2024-09-10", "Election Day": "2024-11-05"}
lo, hi = ax.get_ylim()
pad = (hi - lo) * 0.3            # empty band at the top for the event labels
ax.set_ylim(lo, hi + pad)
for name, d in events.items():
    d = pd.Timestamp(d)
    ax.axvline(d, color="#898781", linewidth=0.8, linestyle=":")
    ax.text(d, hi + pad * 0.95, name, rotation=90, va="top", ha="right", fontsize=8, color="#52514e")

ax.set_title("Weekly issue tweets, May–Nov 2024 (stream plot)", pad=28)
ax.set_xlabel("Week starting")
ax.set_ylabel("Tweets mentioning the issue (centered)")
ax.set_yticks([])  # a centered stream has no meaningful zero; band thickness is the measure
ax.grid(False)
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=5)
fig.tight_layout()
plt.show()

In [ ]:
# Keyword Check: what words actually triggered the issue patterns? This is important for interpreting the results, and for refining the keywords if needed.
# CHECK 1: the full words that triggered each issue's pattern, most common first
trigger_rows = []
for issue in ISSUE_ORDER:
    pattern = re.compile(r"\b\w*(?:" + ISSUE_KEYWORDS[issue] + r")\w*\b")
    hits = text[df[f"issue_{issue}"]].str.findall(pattern).explode().dropna()
    for word, n in hits.value_counts().head(12).items():
        trigger_rows.append({"Issue": ISSUE_LABELS[issue], "Matched word": word, "Tweets": n})

triggers = pd.DataFrame(trigger_rows)
for issue_label, g in triggers.groupby("Issue", sort=False):
    print(f"\n{issue_label}: " + ", ".join(f"{w} ({n})" for w, n in zip(g["Matched word"], g["Tweets"])))

In [ ]:
# CHECK 2a: save 20 random tweets per issue for hand-checking (fill in the 'correct' column with 1 or 0)
CHECK_FILE = "../DataFiles/tk_keyword_check.csv"
sample_rows = []
for issue in ISSUE_ORDER:
    pattern = re.compile(r"\b\w*(?:" + ISSUE_KEYWORDS[issue] + r")\w*\b")
    picks = df[df[f"issue_{issue}"]].sample(20, random_state=42)
    for idx, row in picks.iterrows():
        sample_rows.append({"issue": issue, "week": row["week"],
                            "matched_words": ", ".join(sorted(set(pattern.findall(text[idx])))),
                            "tweet": row["cleanedTweet"], "correct": ""})
#pd.DataFrame(sample_rows).to_csv(CHECK_FILE, index=False) #commented out so it doesnt erase the file if you run this cell again
print(f"Saved {len(sample_rows)} tweets to {CHECK_FILE}. Open it, mark 'correct' as 1 or 0, save, then run the next cell.")

In [ ]:
# CHECK 2b: keyword precision per issue (run after filling in the CSV)
checked = pd.read_csv(CHECK_FILE)
checked["correct"] = pd.to_numeric(checked["correct"], errors="coerce")
done = checked.dropna(subset=["correct"])
if done.empty:
    print("No rows marked yet: fill in the 'correct' column in", CHECK_FILE)
else:
    precision = (done.groupby("issue")["correct"].agg(checked="count", precision="mean")
                     .reindex(ISSUE_ORDER).rename(index=ISSUE_LABELS))
    precision["precision"] = precision["precision"].round(2)
    print(f"Overall keyword precision: {done['correct'].mean():.2f} ({int(done['correct'].sum())} of {len(done)} correct)")
    display(precision)
    print("\nMisses to describe in the report:")
    display(done[done["correct"] == 0][["issue", "matched_words", "tweet"]])